In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

# Masked Softmax
# Padding 위치의 Score
#         ↓ -1e6로 교체
# Softmax 결과가 0
#         ↓
# Padding Token을 Attention에서 제외

In [38]:
# Simple Masked Softmax

# 실제 Token 3개와 Padding 1개의 Attention scores
scores = torch.Tensor([
    [
        2.0,
        1.0,
        3.0,
        5.0,
    ]
])

valid_length = 3

masked_scores = scores.clone()
masked_scores[
    :,
    valid_length:
] = -1e6

attention_weights = nn.functional.softmax(
    masked_scores,
    dim=-1,
)

print(
    "Original Scores:"
)
print(
    scores
)

print(
    "\nMasked Scores:"
)
print(
    masked_scores
)

print(
    "\nAttention Weights:"
)
print(
    attention_weights
)

print(
    "\nWeight Sum:"
)
print(
    attention_weights.sum(
        dim=-1
    )
)

Original Scores:
tensor([[2., 1., 3., 5.]])

Masked Scores:
tensor([[ 2.0000e+00,  1.0000e+00,  3.0000e+00, -1.0000e+06]])

Attention Weights:
tensor([[0.2447, 0.0900, 0.6652, 0.0000]])

Weight Sum:
tensor([1.])


In [ ]:
# Sequence Mask

def sequence_mask(
    X: torch.Tensor,
    valid_lens: torch.Tensor,
    value: float = 0.0,
) -> torch.Tensor:
    
    # X: [N, K]
    # N = Masking할 score row 개수
    # K = 각 row의 key 개수
    
    num_keys = X.shape[1] # K
    
    # [1, K]: [0, 1, ... K-1]
    positions = torch.arange(
        num_keys,
        device=X.device,
    ).reshape(1, -1)
    
    # valid_lens:
    # [N] -> [N, 1]
    #
    # Broadcasting:
    # [1, K] < [N, 1] -> [N, K]
    mask = (
        positions < valid_lens.to(
            device=X.device
        ).reshape(-1, 1)
    )
    
    masked_X = X.clone()
    
    # mask가 False인 위치를 value로 교체
    masked_X[~mask] = value

    return masked_X

In [40]:
# Sequence mask test

X = torch.tensor([
    [10.0, 20.0, 30.0, 40.0],
    [50.0, 60.0, 70.0, 80.0],
    [90.0, 91.0, 92.0, 93.0],
])

valid_lens = torch.tensor([
    2,
    3,
    1,
])

masked_X = sequence_mask(
    X=X,
    valid_lens=valid_lens,
    value=-1e6,
)


print(
    "Original X:"
)
print(
    X
)

print(
    "\nValid Lengths:"
)
print(
    valid_lens
)

print(
    "\nMasked X:"
)
print(
    masked_X
)

Original X:
tensor([[10., 20., 30., 40.],
        [50., 60., 70., 80.],
        [90., 91., 92., 93.]])

Valid Lengths:
tensor([2, 3, 1])

Masked X:
tensor([[ 1.0000e+01,  2.0000e+01, -1.0000e+06, -1.0000e+06],
        [ 5.0000e+01,  6.0000e+01,  7.0000e+01, -1.0000e+06],
        [ 9.0000e+01, -1.0000e+06, -1.0000e+06, -1.0000e+06]])


In [ ]:
# Masked Softmax

def masked_softmax(
    X: torch.Tensor,
    valid_lens: torch.Tensor | None,
) -> torch.Tensor:
    # X shape: [B, Q, K]
    
    if valid_lens is None:
        return nn.functional.softmax(
            X,
            dim=-1,
        )
        
    original_shape = X.shape

    # Batch마다 하나의 valid length
    if valid_lens.ndim == 1:
        
        # [B] -> [B * Q]
        flatten_valid_lens = (
            torch.repeat_interleave(
                valid_lens,
                repeats=original_shape[1], # Q
            )
        )
        
    # Query마다 하나의 valid length
    else:

        # [B, Q] -> [B * Q]        
        flatten_valid_lens = (
            valid_lens.reshape(-1)
        )
        
        
    # [B, Q, K] -> [B * Q, K]
    flattened_X = X.reshape(
        -1,
        original_shape[-1],
    )
    
    
    # Padding score를 -1e6으로 교체
    masked_X = sequence_mask(
        X=flattened_X,
        valid_lens=flatten_valid_lens,
        value=-1e6
    )
    
    # reshape: [B * Q, K] -> [B, Q, K]
    restored_X = masked_X.reshape(
        original_shape
    )
    
    
    # Key dimension을 따라 softmax
    return nn.functional.softmax(
        restored_X,
        dim=-1,
    )

In [42]:
# Masked Softmax with 1D Valid Lengths

torch.manual_seed(42)

# B=2, Q=2, K=4
scores = torch.rand(
    2,
    2,
    4,
)

# Batch 0: 앞의 Key 2개만 유효
# Batch 1: 앞의 Key 3개만 유효
valid_lens_1d = torch.tensor([
    2,
    3,
])

attention_weights_1d = masked_softmax(
    X=scores,
    valid_lens=valid_lens_1d,
)

print(
    "Scores:"
)
print(
    scores
)

print(
    "\nAttention Weights:"
)
print(
    attention_weights_1d
)

print(
    "\nWeight sums:"
)
print(
    attention_weights_1d.sum(
        dim=-1
    )
)

torch.testing.assert_close(
    attention_weights_1d.sum(dim=-1),
    torch.ones(2, 2),
)

Scores:
tensor([[[0.8823, 0.9150, 0.3829, 0.9593],
         [0.3904, 0.6009, 0.2566, 0.7936]],

        [[0.9408, 0.1332, 0.9346, 0.5936],
         [0.8694, 0.5677, 0.7411, 0.4294]]])

Attention Weights:
tensor([[[0.4918, 0.5082, 0.0000, 0.0000],
         [0.4476, 0.5524, 0.0000, 0.0000]],

        [[0.4099, 0.1828, 0.4074, 0.0000],
         [0.3818, 0.2824, 0.3358, 0.0000]]])

Weight sums:
tensor([[1., 1.],
        [1., 1.]])


In [43]:
# Masked Softmax with 2D Valid Lengths

# 각 Query마다 별도의 Valid Length
valid_lens_2d = torch.tensor([
    [1, 3],
    [2, 4],
])

attention_weights_2d = masked_softmax(
    X=scores,
    valid_lens=valid_lens_2d,
)

print(
    "Valid Lengths:"
)
print(
    valid_lens_2d
)

print(
    "\nAttention Weights:"
)
print(
    attention_weights_2d
)

print(
    "\nWeight sums:"
)
print(
    attention_weights_2d.sum(
        dim=-1
    )
)

torch.testing.assert_close(
    attention_weights_2d.sum(dim=-1),
    torch.ones(2, 2),
)

Valid Lengths:
tensor([[1, 3],
        [2, 4]])

Attention Weights:
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.3217, 0.3970, 0.2814, 0.0000]],

        [[0.6916, 0.3084, 0.0000, 0.0000],
         [0.3064, 0.2266, 0.2695, 0.1974]]])

Weight sums:
tensor([[1.0000, 1.0000],
        [1.0000, 1.0000]])


In [44]:
# Batch Matrix Multiplication

# Q shape: [B=2, A=3, D=4]
Q = torch.ones(
    2,
    3,
    4,
)

# K shape: [B=2, D=4, C=6]
K = torch.ones(
    2,
    4,
    6,
)

# [2, 3, 4] @ [2, 4, 6]
# -> [2, 3, 6]
output = torch.bmm(
    Q,
    K,
)

print(
    "Q shape:",
    tuple(Q.shape),
)
print(
    "K shape:",
    tuple(K.shape),
)
print(
    "Output shape:",
    tuple(output.shape),
)

print(
    "\nFirst Batch Output:"
)
print(
    output[0]
)

d2l.check_shape(
    output,
    (
        2,
        3,
        6,
    ),
)

Q shape: (2, 3, 4)
K shape: (2, 4, 6)
Output shape: (2, 3, 6)

First Batch Output:
tensor([[4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4.]])
